# rank0-only-side-effects — worked example 3: Per-rank gradient accumulation vs rank-0 log gate

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank0-only-side-effects`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A common error in distributed code is accidentally placing per-rank computation inside the `if rank == 0:` block, inadvertently making it run only on rank 0. The correct structure has two distinct sections: (1) unlocked code that every rank executes for local computation, and (2) a `if rank == 0:` block exclusively for shared-resource writes like checkpointing and logging.

## Worked solution

**Step 1 — Every rank accumulates its gradient norm.** We compute `grad_norm = compute_grad_norm(rank)` outside any guard. All `world_size` ranks do this.

**Step 2 — Rank 0 only: average across ranks and log.** The average is `sum_of_norms / world_size`. We only compute and log this on rank 0 (`if rank == 0:`).

**Step 3 — Every rank records that it ran.** We use a `ran` list to confirm all 4 ranks ran the per-rank part.

**Step 4 — Verify the correct structure.** The 'ran' count equals `world_size`; the 'logged' count equals 1.

In [ ]:
import torch as t

def mock_compute_grad_norm(rank: int) -> float:
    """Simulate per-rank gradient norm (different per rank)."""
    return 1.0 + rank * 0.1

def distributed_step(rank: int, world_size: int,
                     all_norms_sum: float, ran_list: list, log_list: list) -> None:
    # --- Every rank: compute local grad norm ---
    local_norm = mock_compute_grad_norm(rank)
    ran_list.append(rank)

    # --- Rank 0 only: log the averaged norm ---
    if rank == 0:
        avg_norm = all_norms_sum / world_size
        log_list.append(f'avg_grad_norm={avg_norm:.3f}')

# Simulate 4 ranks
world_size = 4
all_norms_sum = sum(mock_compute_grad_norm(r) for r in range(world_size))

ran = []
logged = []
for rank in range(world_size):
    distributed_step(rank, world_size, all_norms_sum, ran, logged)

print(f'Ranks that ran per-rank code: {ran}')    # [0, 1, 2, 3]
print(f'Log calls: {len(logged)} (expect 1)')    # 1
print(f'Log entry: {logged[0]}')                  # avg_grad_norm=...